# Parts 9–10 — Error Analysis, Evidence Safety, and Deployment Linkage

This notebook inspects concrete failures rather than treating an aggregate score as sufficient. It also verifies that the evaluated pipeline is connected to the local FastAPI/React deployment required by Part 10.

**Scope:** the 30-case audit describes the original pipeline. Its categories are rule-assigned development diagnostics and require independent human review.


In [1]:
from collections import Counter
from pathlib import Path
import csv
import hashlib
import html
import json
import platform
import random
import statistics

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SEED = 20250816
random.seed(SEED)
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

def load_json(relative_path):
    return json.loads((ROOT / relative_path).read_text(encoding="utf-8"))

def print_table(rows, columns):
    if not rows:
        print("(no rows)")
        return
    widths = {
        column: max(len(str(column)), *(len(str(row.get(column, ""))) for row in rows))
        for column in columns
    }
    print(" | ".join(str(column).ljust(widths[column]) for column in columns))
    print("-+-".join("-" * widths[column] for column in columns))
    for row in rows:
        print(" | ".join(str(row.get(column, "")).ljust(widths[column]) for column in columns))

def write_bar_svg(filename, values, title, *, maximum=None):
    values = list(values)
    width, left, right, row_height = 820, 245, 80, 34
    height = 76 + row_height * len(values)
    plot_width = width - left - right
    largest = maximum or max((float(value) for _, value in values), default=1.0) or 1.0
    elements = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="white"/>',
        f'<text x="{width / 2}" y="27" text-anchor="middle" font-family="Arial" font-size="18" font-weight="700">{html.escape(title)}</text>',
    ]
    for index, (label, value) in enumerate(values):
        y = 52 + index * row_height
        bar_width = plot_width * float(value) / largest
        elements.extend([
            f'<text x="{left - 10}" y="{y + 17}" text-anchor="end" font-family="Arial" font-size="13">{html.escape(str(label))}</text>',
            f'<rect x="{left}" y="{y}" width="{bar_width:.2f}" height="20" rx="3" fill="#1c5b58"/>',
            f'<text x="{min(left + bar_width + 7, width - 58):.2f}" y="{y + 16}" font-family="Arial" font-size="12">{float(value):.4g}</text>',
        ])
    elements.append('</svg>')
    target = FIGURES / filename
    target.write_text("\n".join(elements) + "\n", encoding="utf-8")
    print(f"Saved visualization: {target.relative_to(ROOT)}")
    return target

print(f"Project: {ROOT.name} | Python: {platform.python_version()} | fixed seed: {SEED}")


Project: h | Python: 3.12.13 | fixed seed: 20250816


## Error-analysis method

A failure record is included when the verified gold passage is absent from the final top ten of the original system. Each record retains the query ID, query type, Roman query, gold passage ID, top retrieved passage/title, assigned category, failure description, affected system, and a possible future improvement. The proposed improvement field is a hypothesis, not evidence that the problem has been fixed.

The audit is stratified to contain five examples in each of six categories. Category frequencies therefore describe the audit design—not real-world error prevalence.


In [2]:
with (ROOT / "reports/error_analysis/failures_30.csv").open(encoding="utf-8-sig", newline="") as handle:
    failures = list(csv.DictReader(handle))
categories = Counter(row["category"] for row in failures)
statuses = Counter(row["categorization_status"] for row in failures)
print("Failure records:", len(failures))
print("Review status:", dict(statuses))
print_table([{"category": name, "cases": count} for name, count in sorted(categories.items())], ["category", "cases"])
assert len(failures) == 30 and all(count == 5 for count in categories.values())
write_bar_svg("failure_categories.svg", sorted(categories.items()), "Original pipeline: stratified failure audit")


Failure records: 30
Review status: {'rule_assigned_requires_human_review': 30}
category                 | cases
-------------------------+------
Roman spelling mismatch  | 5    
code-switching failure   | 5    
excessive spelling noise | 5    
irrelevant retrieval     | 5    
named-entity mismatch    | 5    
short ambiguous query    | 5    
Saved visualization: reports\figures\failure_categories.svg


![Failure categories](../reports/figures/failure_categories.svg)


## Traceable example from every category


In [3]:
examples = []
for category in sorted(categories):
    row = next(item for item in failures if item["category"] == category)
    examples.append({"category": category, "query_id": row["query_id"], "roman_query": row["roman_urdu_query"], "retrieved_title": row["top_retrieved_title"], "future_hypothesis": row["possible_future_improvement"]})
print_table(examples, ["category", "query_id", "roman_query", "retrieved_title", "future_hypothesis"])


category                 | query_id   | roman_query               | retrieved_title                            | future_hypothesis                                              
-------------------------+------------+---------------------------+--------------------------------------------+----------------------------------------------------------------
Roman spelling mismatch  | raabta-002 | mshil shokd kia hy        | سراج الدین ندوی                            | Learn spelling variants from reviewed Roman-Urdu pairs.        
code-switching failure   | raabta-005 | what is lndo              | راجہ بھوج بین الاقوامی ہوائی اڈہ           | Detect language per token and preserve English entities/terms. 
excessive spelling noise | raabta-011 | bhadri kia h              | لدھےخیل                                    | Add a confidence-aware character-level normalizer.             
irrelevant retrieval     | raabta-009 | sif almlok (ktab) kya hai | مبارکپور                                   | Im

### Interpretation by failure type

- **Roman spelling mismatch / excessive noise:** exact transliteration and word-token retrieval fail when vowels or consonants differ. Character title n-grams directly target this issue.
- **Named-entity mismatch:** semantic retrieval can prefer a related person/place. Title alignment provides a stronger identity signal.
- **Code switching:** English tokens must be preserved while Urdu tokens are normalized; a single-language conversion can damage the query.
- **Short ambiguity:** retrieval should request clarification when several intents remain plausible.
- **Irrelevant retrieval:** a high semantic score does not prove the requested relation. Source and sentence validation must be separate from ranking.


## Evidence-grounding smoke test


In [4]:
smoke = load_json("reports/tables/grounded_qa_smoke.json")
print("Query:", smoke["query"])
print("Supported:", smoke["supported"])
print("Source title:", smoke["source_title"])
print("Source URL:", smoke["source_url"])
print("Abstention reason:", smoke["abstention_reason"])
print("Evidence count:", len(smoke["evidence"]))
for evidence in smoke["evidence"]:
    print(f'- passage={evidence["passage_id"]} similarity={evidence["similarity"]:.4f} text={evidence["text"]}')
assert smoke["supported"] and smoke["evidence"] and smoke["source_url"]


Query: pakistan ka capital kya hai
Supported: True
Source title: پاکستان کے دارالحکومت
Source URL: https://ur.wikipedia.org/wiki/%D9%BE%D8%A7%DA%A9%D8%B3%D8%AA%D8%A7%D9%86%20%DA%A9%DB%92%20%D8%AF%D8%A7%D8%B1%D8%A7%D9%84%D8%AD%DA%A9%D9%88%D9%85%D8%AA
Abstention reason: None
Evidence count: 2
- passage=58191-p0000-b39fd7c03fbb similarity=0.8783 text=یہ پاکستان کے قومی اور صوبائی دارالحکومتوں کی ایک فہرست ہے۔
- passage=58191-p0000-b39fd7c03fbb similarity=0.8449 text=قومی دار الحکومت 1960ء سے پاکستان کا قومی یا وفاقی دار الحکومت اسلام آباد ہے۔


The answer is extractive and tied to one source. The application additionally checks reranker relevance, converted-term/title alignment, sentence similarity, and requested answer shape. Date, birth/death, price, quantity, and current-capital questions have relation-specific rules. If no candidate passes, the system returns an explicit abstention reason with no invented evidence.


## Part 10 — Deployment linkage and local completeness


In [5]:
required_paths = ["backend/app/main.py", "backend/app/models.py", "backend/app/service.py", "frontend/package.json", "frontend/src/App.tsx", "frontend/dist/index.html"]
deployment_rows = [{"path": path, "present": (ROOT / path).is_file()} for path in required_paths]
print_table(deployment_rows, ["path", "present"])
audit = load_json("reports/tables/portability_audit.json")
print("Backend sources audit:", audit["checks"]["backend_sources_present"]["passed"])
print("Frontend build audit:", audit["checks"]["frontend_build_present"]["passed"])
print("Portable relative paths audit:", audit["checks"]["no_machine_specific_source_paths"]["passed"])
assert all(row["present"] for row in deployment_rows)


path                     | present
-------------------------+--------
backend/app/main.py      | True   
backend/app/models.py    | True   
backend/app/service.py   | True   
frontend/package.json    | True   
frontend/src/App.tsx     | True   
frontend/dist/index.html | True   
Backend sources audit: True
Frontend build audit: True
Portable relative paths audit: True


The deployable application is not implemented inside the notebook. Part 10 is satisfied by the repository's FastAPI backend and React/Vite frontend, which consume the same retrieval/evidence pipeline measured here. The interface exposes query transformations, route contributions, candidate counts, confidence, evidence gates, source URL, abstention reason, and component latency.

## Final analysis conclusion

The original failure audit explains the user's reported symptom: related but incorrect results were passing through a retrieval-centric pipeline. The romanized-title route materially improves candidate coverage, while source/relation gates reduce unsupported answers. Remaining work is independent annotation review, post-change category analysis, confidence calibration, clarification for ambiguity, broader Urdu collections, and the one-time locked-test evaluation after configuration freeze.
